In [1]:
import torch as t
import pandas as pd
import sys
import os
import json
import itertools
import numpy as np
import yaml
sys.path.append("../")
device = t.device("cuda" if t.cuda.is_available() else "cpu")

In [2]:
with open('../configs/generation_config.yaml', 'r') as file:
    generation_config = yaml.safe_load(file)
        
with open('../psychometric_tests/hexaco_100_eval.yaml', 'r') as file:
    hexaco_eval = yaml.safe_load(file)

In [3]:
def write_to_json(file, file_path):
    with open(file_path, 'w') as f:
        json.dump(file, f)
        
def read_json(file_path):
    with open(file_path, "r") as f:
        file = json.load(f)
    return file

In [4]:
def get_refusal_rate(answers):  
    answers = pd.Series(answers)
    refusal_answers = answers[answers.isin(['Do not wish to answer','Do not wish to answer.'])]
    return np.round(len(refusal_answers)/len(answers),3).item()

def get_mean_metrics(input_list):
    
    list_mean = np.round(np.mean(input_list), 3).item()
    list_std = np.round(np.std(input_list), 3).item()
    return list_mean, list_std

In [15]:
def get_refusal_rate_dict(data_dir):
    refusal_rate_dict = {}
    for filename in os.listdir(data_dir):
        if ".json" in filename:
            answers = read_json(os.path.join(data_dir,filename))
            for i, answer in enumerate(answers):
                if f"persona_{i}" not in refusal_rate_dict.keys():
                    refusal_rate_dict[f'persona_{i}'] = {}

                refusal_rate = get_refusal_rate(list(itertools.chain(*answer['answers'])))    
                refusal_rate_dict[f'persona_{i}'][filename.split(".")[0]] = refusal_rate
    
    return refusal_rate_dict

In [16]:
pd.DataFrame(get_refusal_rate_dict("persona_basic_info_experiment_results_v2"))

,persona_0
paraphrase_hexaco_answers_llama32_1b_it,0.164
paraphrase_hexaco_answers_gpt_41_mini,0.020
normal_hexaco_answers_without_no_llama32_1b_it,0.000
normal_hexaco_answers_without_no_gpt_41_mini,0.000
paraphrase_hexaco_answers_without_no_llama32_1b_it,0.000
normal_hexaco_inverted_likert_without_no_answers_gpt_41_mini,0.000
paraphrase_hexaco_answers_without_no_gpt_41_mini,0.000
paraphrase_hexaco_inverted_likert_without_no_answers_llama32_1b_it,0.000
paraphrase_hexaco_inverted_likert_answers_llama32_1b_it,0.198
paraphrase_hexaco_inverted_likert_answers_gpt_41_mini,0.023


In [7]:
def calculate_hexaco_score(trait, subtrait, answers, likert_scale = generation_config['likert_scale']):
    answer_dict = {}
    answers = [answer if answer in likert_scale else "Do not wish to answer" for answer in answers]
    answers = pd.Series(answers)
    subtrait_dict = hexaco_eval[trait][subtrait]
    indices = [idx - 1 for idx in subtrait_dict['indices']]
    trait_answers = answers[indices]
    refused_answers = trait_answers[trait_answers.isin(['Do not wish to answer','Do not wish to answer.'])]
    non_refused_answers = trait_answers[~trait_answers.isin(['Do not wish to answer','Do not wish to answer.'])]
    answer_indices = [likert_scale.index(answer) for answer in non_refused_answers]
    true_answer_indices = [6-idx if reverse else idx for idx,reverse in zip(answer_indices, subtrait_dict['reverse'])]
    
    answer_dict['answer_indices'] = answer_indices
    answer_dict['true_answer_indices'] = true_answer_indices
    answer_dict['n_answered_questions'] = len(true_answer_indices)
    answer_dict['n_refused_questions'] = len(refused_answers)
    answer_dict['trait'] = trait
    answer_dict['subtrait'] = subtrait
    answer_dict['subtrait_score'] = np.round(np.mean(true_answer_indices).item(),3)
    return answer_dict

def get_trait_scores(iteration, persona,answer, likert_scale,filename):
    subtrait_hexaco_scores = []
    trait_hexaco_scores = []
    for trait in hexaco_eval.keys():
        trait_hexaco_score = {}
        trait_hexaco_score['persona'] = persona
        trait_hexaco_score['iteration'] = iteration
        trait_hexaco_score['trait'] = trait
        trait_hexaco_score['true_answer_indices'] = []
        trait_hexaco_score['n_answered_questions'] = 0
        trait_hexaco_score['n_refused_questions'] = 0
        for subtrait in hexaco_eval[trait].keys():
            subtrait_hexaco_score = calculate_hexaco_score(trait, subtrait, answer, likert_scale)
            subtrait_hexaco_score['persona'] = persona
            subtrait_hexaco_score['iteration'] = iteration
            trait_hexaco_score['true_answer_indices'].extend(subtrait_hexaco_score['true_answer_indices'])
            trait_hexaco_score['n_answered_questions'] += subtrait_hexaco_score['n_answered_questions']
            trait_hexaco_score['n_refused_questions'] += subtrait_hexaco_score['n_refused_questions']
            if "inverted" in filename:
                subtrait_hexaco_score['likert_scale'] = "inverse"
            else:
                subtrait_hexaco_score['likert_scale'] = "normal"
            if "paraphrase" in filename:
                subtrait_hexaco_score['paraphrase'] = "paraphrase"
            else:
                subtrait_hexaco_score['paraphrase'] = "normal"
            if "without_no" in filename:
                subtrait_hexaco_score['refusal_allowed'] = "No Refusal"
            else:
                subtrait_hexaco_score['refusal_allowed'] = "Refusal"
            if "llama" in filename:
                subtrait_hexaco_score['model'] = "llama_3.2_1b_it"
            else:
                subtrait_hexaco_score['model'] = "gpt_4.1_mini"
                
            subtrait_hexaco_scores.append(subtrait_hexaco_score)
        trait_hexaco_score['trait_score'] = np.round(np.mean(trait_hexaco_score['true_answer_indices']).item(),3)
        if "inverted" in filename:
            trait_hexaco_score['likert_scale'] = "inverse"
        else:
            trait_hexaco_score['likert_scale'] = "normal"
        if "paraphrase" in filename:
            trait_hexaco_score['paraphrase'] = "paraphrase"
        else:
            trait_hexaco_score['paraphrase'] = "normal"
        if "without_no" in filename:
            trait_hexaco_score['refusal_allowed'] = "No Refusal"
        else:
            trait_hexaco_score['refusal_allowed'] = "Refusal"
        if "llama" in filename:
            trait_hexaco_score['model'] = "llama_3.2_1b_it"
        else:
            trait_hexaco_score['model'] = "gpt_4.1_mini"
        trait_hexaco_scores.append(trait_hexaco_score)
        
    return pd.DataFrame(trait_hexaco_scores), pd.DataFrame(subtrait_hexaco_scores)

In [8]:
def get_all_stats(data_dir):
    trait_df = pd.DataFrame()
    subtrait_df = pd.DataFrame()
    for filename in os.listdir(data_dir):
        print(filename)
        persona_answers = read_json(os.path.join(data_dir,filename))
        for answers in persona_answers:
            for i,answer in enumerate(answers['answers']):
                iteration_trait_df, iteration_subtrait_df = get_trait_scores(i,answers['config']['persona'],answer, generation_config['likert_scale'],filename)
                subtrait_df = pd.concat([subtrait_df,iteration_subtrait_df], axis = 0)
                trait_df = pd.concat([trait_df,iteration_trait_df], axis = 0)
    
    subtrait_df_grouped = subtrait_df.groupby(['trait','subtrait','persona','likert_scale','paraphrase','refusal_allowed','model'],as_index = False).\
    agg({"n_answered_questions":"sum","n_refused_questions":"sum","subtrait_score":["mean",np.std]})
    
    subtrait_df_grouped.columns = ['trait','subtrait','persona','likert_scale','paraphrase','refusal_allowed','model','n_answered_questions','n_refused_questions',"subtrait_score_mean","subtrait_score_std"]
    
    trait_df_grouped = trait_df.groupby(['trait','persona','likert_scale','paraphrase','refusal_allowed','model'],as_index = False).\
    agg({"n_answered_questions":"sum","n_refused_questions":"sum","trait_score":["mean",np.std]})
    
    trait_df_grouped.columns = ['trait','persona','likert_scale','paraphrase','refusal_allowed','model','n_answered_questions','n_refused_questions',"trait_score_mean","trait_score_std"]
    
    subtrait_df_grouped.loc[:,'refusal_rate'] = subtrait_df_grouped.loc[:,'n_refused_questions']/(subtrait_df_grouped.loc[:,'n_answered_questions'] + subtrait_df_grouped.loc[:,'n_refused_questions'])
    trait_df_grouped.loc[:,'refusal_rate'] = trait_df_grouped.loc[:,'n_refused_questions']/(trait_df_grouped.loc[:,'n_answered_questions'] + trait_df_grouped.loc[:,'n_refused_questions'])

    return subtrait_df_grouped, trait_df_grouped
    

In [9]:
subtrait_df_grouped, trait_df_grouped = get_all_stats("persona_basic_info_experiment_results_v2")

paraphrase_hexaco_answers_llama32_1b_it.json


/Users/shreyansjain/.pyenv/versions/3.11.9/envs/llm_psychometrics/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/shreyansjain/.pyenv/versions/3.11.9/envs/llm_psychometrics/lib/python3.11/site-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/Users/shreyansjain/.pyenv/versions/3.11.9/envs/llm_psychometrics/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/shreyansjain/.pyenv/versions/3.11.9/envs/llm_psychometrics/lib/python3.11/site-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/Users/shreyansjain/.pyenv/versions/3.11.9/envs/llm_psychometrics/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:3596: 

paraphrase_hexaco_answers_gpt_41_mini.json
normal_hexaco_answers_without_no_llama32_1b_it.json
normal_hexaco_answers_without_no_gpt_41_mini.json
paraphrase_hexaco_answers_without_no_llama32_1b_it.json
normal_hexaco_inverted_likert_without_no_answers_gpt_41_mini.json
paraphrase_hexaco_answers_without_no_gpt_41_mini.json
paraphrase_hexaco_inverted_likert_without_no_answers_llama32_1b_it.json
paraphrase_hexaco_inverted_likert_answers_llama32_1b_it.json


/Users/shreyansjain/.pyenv/versions/3.11.9/envs/llm_psychometrics/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/shreyansjain/.pyenv/versions/3.11.9/envs/llm_psychometrics/lib/python3.11/site-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/Users/shreyansjain/.pyenv/versions/3.11.9/envs/llm_psychometrics/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/shreyansjain/.pyenv/versions/3.11.9/envs/llm_psychometrics/lib/python3.11/site-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/Users/shreyansjain/.pyenv/versions/3.11.9/envs/llm_psychometrics/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:3596: 

paraphrase_hexaco_inverted_likert_answers_gpt_41_mini.json
normal_hexaco_inverted_likert_answers_gpt_41_mini.json
normal_hexaco_inverted_likert_without_no_answers_llama32_1b_it.json
normal_hexaco_inverted_likert_answers_llama32_1b_it.json


/Users/shreyansjain/.pyenv/versions/3.11.9/envs/llm_psychometrics/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/shreyansjain/.pyenv/versions/3.11.9/envs/llm_psychometrics/lib/python3.11/site-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/Users/shreyansjain/.pyenv/versions/3.11.9/envs/llm_psychometrics/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/shreyansjain/.pyenv/versions/3.11.9/envs/llm_psychometrics/lib/python3.11/site-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/Users/shreyansjain/.pyenv/versions/3.11.9/envs/llm_psychometrics/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:3596: 

normal_hexaco_answers_llama32_1b_it.json


/Users/shreyansjain/.pyenv/versions/3.11.9/envs/llm_psychometrics/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/shreyansjain/.pyenv/versions/3.11.9/envs/llm_psychometrics/lib/python3.11/site-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/Users/shreyansjain/.pyenv/versions/3.11.9/envs/llm_psychometrics/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/shreyansjain/.pyenv/versions/3.11.9/envs/llm_psychometrics/lib/python3.11/site-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/Users/shreyansjain/.pyenv/versions/3.11.9/envs/llm_psychometrics/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:3596: 

normal_hexaco_answers_gpt_41_mini.json
paraphrase_hexaco_inverted_likert_without_no_answers_gpt_41_mini.json


/var/folders/9b/k67tngbx13jgzw5kjx9_d0tc0000gn/T/ipykernel_67752/2077101286.py:14: FutureWarning: The provided callable <function std at 0x10decee80> is currently using SeriesGroupBy.std. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "std" instead.
  agg({"n_answered_questions":"sum","n_refused_questions":"sum","subtrait_score":["mean",np.std]})
/var/folders/9b/k67tngbx13jgzw5kjx9_d0tc0000gn/T/ipykernel_67752/2077101286.py:19: FutureWarning: The provided callable <function std at 0x10decee80> is currently using SeriesGroupBy.std. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "std" instead.
  agg({"n_answered_questions":"sum","n_refused_questions":"sum","trait_score":["mean",np.std]})


In [10]:
gpt_subtrait_stat_df = subtrait_df_grouped[subtrait_df_grouped['model'] == "gpt_4.1_mini"]
llama_subtrait_stat_df = subtrait_df_grouped[subtrait_df_grouped['model'] == "llama_3.2_1b_it"]

gpt_trait_stat_df = trait_df_grouped[trait_df_grouped['model'] == "gpt_4.1_mini"]
llama_trait_stat_df = trait_df_grouped[trait_df_grouped['model'] == "llama_3.2_1b_it"]

In [11]:
gpt_subtrait_stat_pivot_df = gpt_subtrait_stat_df.pivot(values = ['refusal_rate','subtrait_score_mean','subtrait_score_std',], columns=['paraphrase', 'likert_scale','refusal_allowed'], index=['persona','trait','subtrait'])

gpt_trait_stat_pivot_df = gpt_trait_stat_df.pivot(values = ['refusal_rate','trait_score_mean','trait_score_std',], columns=['paraphrase', 'likert_scale','refusal_allowed'], index=['persona','trait'])

In [12]:
llama_subtrait_stat_pivot_df = llama_subtrait_stat_df.pivot(values = ['refusal_rate','subtrait_score_mean','subtrait_score_std',], columns=['paraphrase', 'likert_scale','refusal_allowed'], index=['persona','trait','subtrait'])

llama_trait_stat_pivot_df = llama_trait_stat_df.pivot(values = ['refusal_rate','trait_score_mean','trait_score_std',], columns=['paraphrase', 'likert_scale','refusal_allowed'], index=['persona','trait'])

In [13]:
gpt_trait_stat_pivot_df.to_csv(os.path.join("persona_basic_info_experiment_results_v2","gpt_trait_stat_pivot_df.csv"))
llama_trait_stat_pivot_df.to_csv(os.path.join("persona_basic_info_experiment_results_v2","llama_trait_stat_pivot_df.csv"))
gpt_subtrait_stat_pivot_df.to_csv(os.path.join("persona_basic_info_experiment_results_v2","gpt_subtrait_stat_pivot_df.csv"))
llama_subtrait_stat_pivot_df.to_csv(os.path.join("persona_basic_info_experiment_results_v2","llama_subtrait_stat_pivot_df.csv"))